# Practical 2 - Kinematics
In this practical, you will calculate the forward kinematics and implement differential inverse kinematics for the UR3e robot arm. To complete the specified functions in the practical, see the [type hints](https://github.com/airo-ugent/airo-mono/blob/main/airo-typing/airo_typing/__init__.py) for the expected input and output types.

In [ ]:
# Let's first make numpy outputs more readable 
import numpy as np
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# Connect to the real robot 
from airo_typing import JointConfigurationType, HomogeneousMatrixType, RotationMatrixType
from airo_robots.manipulators.hardware.ur_rtde import URrtde

import math

ip_address = "10.42.0.162"  # fill in the robot IP
robot = URrtde(ip_address, URrtde.UR3E_CONFIG)

## Part 1 - Forward kinematics

The parameters of the UR3e robot are defined by the following [DH table](https://www.universal-robots.com/articles/ur/application-installation/dh-parameters-for-calculations-of-kinematics-and-dynamics/) (standard DH convention):

|  Joint |  $\theta$ [rad] |  $d$ [m] |   $a$ [m] |    $\alpha$ [rad] |
|-------:|----------------:|---------:|----------:|------------------:|
|      1 |    $\theta_1^*$ |  0.15185 |        0. |   $\frac{\pi}{2}$ |
|      2 |    $\theta_2^*$ |       0. |  -0.24355 |                0. |
|      3 |    $\theta_3^*$ |       0. |   -0.2132 |                0. |
|      4 |    $\theta_4^*$ |  0.13105 |        0. |   $\frac{\pi}{2}$ |
|      5 |    $\theta_5^*$ |  0.08535 |        0. |  $\frac{-\pi}{2}$ |
|      6 |    $\theta_6^*$ |   0.0921 |        0. |                0. |

The above table is only valid for the frames assigned as follows:

![UR3e frames](https://www.universal-robots.com/media/1803001/images.jpg?width=800&height=374.3680188124633)

Also, from the theory session you know that the transition from frame $i-1$ to frame $i$ is then defined by $^{i-1}T^{i} = \text{Rot}(z,\theta)\text{Trans}(z,d)\text{Trans}(x,a)\text{Rot}(x,\alpha)$.

<div class="alert alert-block alert-info">  <b>Task 1</b>: Complete the function `forward_kinematics_ur3e(q)` where $q=[\theta_1,\theta_2,\theta_3,\theta_4,\theta_5,\theta_6]$. The function should return $^BX^{TCP}$. </div>


In [ ]:
def forward_kinematics_ur3e(q: JointConfigurationType) -> HomogeneousMatrixType:
    ...
    return X_B_TCP

We will now test `forward_kinematics_ur3e` on the robot.

<div class="alert alert-block alert-info">  <b>Task 2</b>: sample some well-chosen joint configurations and run them on the robot. Use the `X_B_TCP = robot.get_tcp_pose()` to compare. </div>

In [ ]:
q_to_test = [0., -np.pi / 2, 0., -np.pi / 2, 0., 0.]  # sample multiple joint configurations!
robot.move_to_joint_configuration(q_to_test).wait()
X_B_TCP_real = robot.get_tcp_pose()
X_B_TCP_virtual = forward_kinematics_ur3e(q_to_test)
print(X_B_TCP_virtual - X_B_TCP_real)

<div class="alert alert-block alert-info">  <b>Task 3</b>: Which points in space did you evaluate in order to get a good view on whether your forward kinematics function works or not?</div>

Your answer: 

<div class="alert alert-block alert-info">  <b>Task 4</b>: Give possible causes for small errors.</div>

Your answer:

## Part 2 - Inverse kinematics
As you know, the forward kinematics is given by a homogeneous transformation matrix from the base of the robot arm to the end-effector. In order to find the joints for a desired pose , one needs to solve these equations. However, as the number of joints and the complexity of the arm's geometry increase, finding an analytical solution becomes increasingly challenging. However, one can rewrite this problem as an optimization problem. Therefore, we consider the error between the desired and actual pose. In order to do this, we need an expression that gives us the relation between the desired speed of the end-effector and the speed at which we need to change the joints. This relation is captured by the Jacobian. To solve the inverse kinematics problem, we can apply the following algorithm:

1. Calculate the error $e$ between the actual and desired position;
2. Compute the Jacobian $J(q)$;
3. Transpose the Jacobian $J^{\text{*}}$;
4. Calculate $\Delta \theta = J^{\text{*}} e$;
5. Move $\theta$: $\theta = \theta + \alpha \Delta \theta$, with $\alpha$ the step size;
6. repeat 1-5 until the error gets sufficiently small.

Here, $J^{\text{*}}$ can be the transposed Jacobian or the pseudo-inverse of the Jacobian. In the theory session the Jacobian was defined as 

$J=\begin{bmatrix}
J_v\\
J_\omega
\end{bmatrix}$,

in the practicals we switch the positions of the rotational and translational part of the Jacobian: 

$J=\begin{bmatrix}
J_\omega\\
J_v
\end{bmatrix}$.

The reason for first specifying the angular part is because many robotics libraries first specify rotations, followed by positions (e.g. you do this yourself when building a homogenous matrix).  
This order will be important when determining the form of the error vector $e$, we will come back to this further on.



### Importing the Jacobian from a URDF file

In the theory session you saw how to compute the Jacobian. While you can do this as an optional (time-consuming) exercise, in this practical session we will derive the Jacobian from the robot kinematics defined by a URDF file. You will learn more about URDF - Unified Robot Description Format - in module 3 when we talk about planning. For now, you should know that URDF is an alternative specification for a robot to the DH notation you already saw. In short, URDF is an XML-based file format used to describe the kinematic and dynamic properties of a robot in a standardised way. Apart from the robot description, URDF files can also be used to describe the environment.

In the following cells we build an environment in a robotic simulator called [Drake](https://drake.mit.edu). From Drake we can easily obtain the Jacobian, and - as you will see in module 3 - also all kinds of other information (e.g., collisions). You don't have to go through this code in detail. We will explain some high-level concepts along the way. You do not need these but it helps to understand what is happening in robotic simulators.

In [ ]:
import os

from pydrake.geometry import MeshcatVisualizer, Meshcat
from pydrake.multibody.plant import AddMultibodyPlantSceneGraph
from pydrake.systems.framework import DiagramBuilder
from pydrake.multibody.parsing import Parser
from pydrake.multibody.plant import MultibodyPlant
from pydrake.multibody.tree import JacobianWrtVariable

import airo_models 

meshcat = Meshcat()

[Meshcat](https://github.com/meshcat-dev/meshcat) is a 3D viewer in which we can view the simulation environment. Open the Drake Meshcat environment in the browser by clicking the localhost link in the output of the cell above. Here, your so-called *digital twin* will be shown.

The following cell will create the simulation environment, populate it with the UR3e robot and show it in Meshcat.

In [ ]:
# Let's make sure the meshcat visualization is clean
meshcat.Delete()

# Stitch together the robot, simulator and visualization 
builder = DiagramBuilder()
# Create the simulator and scene graph
plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0)
parser = Parser(plant)
# Here we add the robot to the simulator. The robot is defined with an URDF file instead of a DH table. 
ur3e_urdf_fp = airo_models.get_urdf_path("ur3e")
ur3e_idx = parser.AddModels(ur3e_urdf_fp)[0]

# We explicitely fix the robot to the world. If we do not make this weld, we imply the robot is a mobile robot. 
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_link", ur3e_idx)
)
plant.Finalize()

MeshcatVisualizer.AddToBuilder(
    builder, scene_graph.get_query_output_port(), meshcat
)

diagram = builder.Build()

# Make a data container object (a context) for the robot. This will store all data associated with the simulation (positions, rotations, velocities etc).
robot_context = diagram.CreateDefaultContext()
diagram.ForcedPublish(robot_context)

<div class="alert alert-block alert-success"> Your robot should be shown in the MeshCat server, open the web page to see the simulation.</div>

The following terminology is used in the context of the Drake Robotics simulator:
<ul>
  <li><b>Plant</b>: alternative name for a simulation.</li>
  <li><b>Context</b>: data container. We need this because the Plant (== simulation) object is stateless.</li>
  <li><b>Diagram</b>: set of interconnected systems that communicate with each other.</li>
</ul>

Next, we will define three frames we often use in manipulation: the TCP frame, the robot base frame and the world origin frame. 

In [ ]:
TCP_frame = plant.GetFrameByName("tool0")
B_frame = plant.GetFrameByName("base")
W_frame = plant.world_frame()

##### Frame vs. pose
As a small digression, we will briefly detail the difference between *frame* and *pose*.
A <b>frame</b> is a mathematical object encapsulating a right-handed orthogonal basis attached to a point on a physical body. It allows to calculate a pose (translation + rotation) of this point on the body, relative to other bodies. Hence, the pose of the frame is only meaningful relative to another frame. 

So in summary;

- Pose = translation + rotation between two frames, often represented as a homogenous (4x4) matrix.
- Frame = orthogonal base attached to a point on a body. Has a pose, always relative to another frame.

For example; the pose X_B_TCP gives us the pose of the TCP relative to the base of the robot by:
`B_frame (robot base frame) ----> X_B_TCP (pose) ----> TCP_frame`

Next, we define some visualization functions. This is to help debug your IK method. These are helper functions that draw triads in meshcat. You do not need to understand or read this.

In [ ]:
# This is some visualization logic for in Meshcat

from pydrake.geometry import Cylinder
from pydrake.geometry import Rgba
from pydrake.math import RigidTransform, RotationMatrix


def visualize_tcp_triad(tcp_pose, iteration):
    if iteration < 10:
    # plot first10 iterations
        add_triad(meshcat, f"ik/it{iteration}", opacity=0.20, X_P_T=tcp_pose)
    # only plot every X iterations because it is heavy on the browser
    elif iteration % 50 == 0:
        add_triad(meshcat, f"ik/it{iteration}", opacity=0.20, X_P_T=tcp_pose)


def add_triad(meshcat, path, length=0.05, radius=0.002, opacity=1.0, X_P_T=RigidTransform(), X_W_P=RigidTransform()):
    # We know we are getting poses relative to an arbitrary parent (the base in this practical). However, MeshCat will plot the transforms relative to the world and not to the base of the robot.
    # We change base here
    X_PT = RigidTransform(X_W_P) @ RigidTransform(X_P_T)

    meshcat.SetTransform(path, X_PT)
    # x-axis
    X_TG = RigidTransform(RotationMatrix.MakeYRotation(np.pi / 2), [length / 2.0, 0, 0])
    meshcat.SetTransform(path + "/x-axis", X_TG)
    meshcat.SetObject(path + "/x-axis", Cylinder(radius, length), Rgba(1, 0, 0, opacity))

    # y-axis
    X_TG = RigidTransform(RotationMatrix.MakeXRotation(np.pi / 2), [0, length / 2.0, 0])
    meshcat.SetTransform(path + "/y-axis", X_TG)
    meshcat.SetObject(path + "/y-axis", Cylinder(radius, length), Rgba(0, 1, 0, opacity))

    # z-axis
    X_TG = RigidTransform([0, 0, length / 2.0])
    meshcat.SetTransform(path + "/z-axis", X_TG)
    meshcat.SetObject(path + "/z-axis", Cylinder(radius, length), Rgba(0, 0, 1, opacity))

Next is another helper function to query the simulator for the TCP position relative to the base of the robot: X_B_TCP. You will not have to use this method yourself.

In [ ]:
def get_tcp_pos(plant, plant_context):
    TCP_frame = plant.GetFrameByName("tool0")
    B_frame = plant.GetFrameByName("base")
    X_B_TCP = TCP_frame.CalcPose(plant_context, B_frame)
    return X_B_TCP.GetAsMatrix4()

### Inverse kinematics: the transposed Jacobian method
In this part, you will implement differential IK using the transposed Jacobian. We provide some boilerplate code with helper functions to get you started. 

The first helper function `pose_nearby()` calculates the difference between poses. Distance calculation for positions are trivial but distances between rotations are a bit more involved. 

In [ ]:
# Some helper functions
from scipy.spatial import distance

def delta_rotations(rotA: RotationMatrix, rotB: RotationMatrix):
    rpy_a = rotA.ToRollPitchYaw().vector()
    rpy_b = rotB.ToRollPitchYaw().vector()
    return np.abs(rpy_a - rpy_b)

def rotation_within(rotA: RotationMatrix, rotB: RotationMatrix, within: float):
    return np.all(delta_rotations(rotA, rotB) < within)


def nearby(X_a: RigidTransform, X_b: RigidTransform, pos_tol: float = 0.001, rot_tol: float = math.radians(1)) -> bool:
    """
    Checks if two poses are nearby each other.

    Args:
        X_a: RigidTransform representing the first pose.
        X_b: RigidTransform representing the second pose.
        pos_tol: amount of meters difference allowed.
        rot_tol: amount of radians difference allowed.

    Returns:
        bool: True if the poses are nearby each other, False otherwise.
    """
    within_distance = distance.euclidean(X_a.translation(), X_b.translation()) < pos_tol
    within_rotation = rotation_within(X_a.rotation(), X_b.rotation(), rot_tol)
    return within_distance and within_rotation
    
# Calculate the distance between positions and rotations
def pose_nearby(X_a: HomogeneousMatrixType, X_b: HomogeneousMatrixType, pos_tol: float = 0.001,
                rot_tol: float = math.radians(1)):
    return nearby(RigidTransform(X_a), RigidTransform(X_b), pos_tol, rot_tol)

# Convert a rotation matrix to its roll pitch yaw representation
def rotation_matrix_to_rpy(R: RotationMatrixType):
    return RotationMatrix(R).ToRollPitchYaw().vector()


<div class="alert alert-block alert-info">  <b>Task 5</b>: Complete the algorithm below to implement the transpose Jacobian method for inverse kinematics.</div>

We already provide some logic for you: we give you the current distance between the current and target pose, we give you the jacobian and we use your new joint values to continue the IK loop while visualizing it in the MeshCat browser tab.

We require you to fill step 3, 4 and 5 of the algorithm. See the method `inverse_kinematics_ur3e()` below. We have marked these steps with comment `TODO(@student)`.

Pay close attention to the provided form of the error vector $e$ and validate for yourself that this is correct given the Drake convention that the Jacobian puts the angular part on top of the translational part
$J=\begin{bmatrix}
J_\omega\\
J_v
\end{bmatrix}$.

In [ ]:
def inverse_kinematics_ur3e(X_B_TCP_target, q0, alpha = 0.1, pos_tolerance_in_m=0.001, rot_tolerance_in_degrees=1,
                            plant=plant, robot_context=robot_context, TCP_frame=TCP_frame, B_frame=B_frame):
    all_delta_q = []
    all_jacobians = []
    
    # reset simulation positions to the original ones
    plant.SetPositions(plant.GetMyContextFromRoot(robot_context), ur3e_idx, q0)
    diagram.ForcedPublish(robot_context)

    # q0 to TCP pose by quering the simulator
    X_B_TCP_current = get_tcp_pos(plant, plant.GetMyContextFromRoot(robot_context))
    # We query the current joint values (should be q0) (this variable 'q_calculated' will be used to update the new joint values)
    q_calculated = plant.GetPositions(plant.GetMyContextFromRoot(robot_context), ur3e_idx).reshape((-1, 1))  
    
    # Let's visualize where we come from and where we are going to. We need X_W_B because meshcat will plot triads relative to the world_frame and not the robot_base_frame
    X_W_B = B_frame.CalcPose(plant.GetMyContextFromRoot(robot_context), plant.world_frame())
    add_triad(meshcat, f"ik/X_TCP_current", opacity=1, length=0.10, radius=0.005, X_P_T=X_B_TCP_current, X_W_P=X_W_B)
    add_triad(meshcat, f"ik/X_TCP_target", opacity=1, length=0.10, radius=0.005, X_P_T=X_B_TCP_target, X_W_P=X_W_B)
    
    # numerical estimation of the inverse kinematics based on the jacobian
    nr_iteration = 0
    max_iterations = 10000
    while not pose_nearby(X_B_TCP_target, X_B_TCP_current, pos_tol=pos_tolerance_in_m, rot_tol=math.radians(rot_tolerance_in_degrees)) and nr_iteration < max_iterations:
        # Step 1: calculate the error (don't edit)
        rpy_target = rotation_matrix_to_rpy(X_B_TCP_target[:3, :3])
        rpy_current = rotation_matrix_to_rpy(X_B_TCP_current[:3, :3])
        rpy_error = rpy_target - rpy_current
    
        pos_target = X_B_TCP_target[:3, 3]
        pos_current = X_B_TCP_current[:3, 3]
        pos_error = pos_target - pos_current
    
        error = np.vstack((rpy_error.reshape((3, 1)), pos_error.reshape((3, 1))))
        
        # Step 2: compute the jacobian (don't edit)
        J = plant.CalcJacobianSpatialVelocity(
            plant.GetMyContextFromRoot(robot_context),
            JacobianWrtVariable.kQDot,
            TCP_frame,
            [0, 0, 0],
            B_frame,
            B_frame
        )
        all_jacobians.append(J)
        
        # Step 3: transpose the jacobian: TODO(@student)
        J_transposed = ...
    
        # Step 4: calculate joint deltas: TODO(@student)
        delta_q = ...
        all_delta_q.append(delta_q.flatten())  # We store the joint delta vectors calculated at each step to visualize their convergence later on. 
        
        # Step 5: update the joint values (q_calculated means the new joint values): TODO(@student)
        q_calculated = ...
    
        # Set the simulated robot at the new joint position (don't edit the remainder of the code in this cell)
        plant.SetPositions(plant.GetMyContextFromRoot(robot_context), ur3e_idx, q_calculated)
    
        # Get tcp pose now that joints are updated (cfr. Forward Kinematics)
        X_B_TCP_current = get_tcp_pos(plant, plant.GetMyContextFromRoot(robot_context))
    
        visualize_tcp_triad(TCP_frame.CalcPose(plant.GetMyContextFromRoot(robot_context), plant.world_frame()), nr_iteration)
        diagram.ForcedPublish(robot_context)
        nr_iteration += 1

    if not pose_nearby(X_B_TCP_target, X_B_TCP_current, pos_tol=pos_tolerance_in_m, rot_tol=math.radians(rot_tolerance_in_degrees)):
        print(f"IK did not converge after {nr_iteration} iters!")
    
    return q_calculated, all_delta_q, all_jacobians

In [ ]:
# Define target pose to test your completed IK function
X_B_TCP_target = np.identity(4) 
X_B_TCP_target[0:3, 3] = np.array([-0.25, -0.05, 0.45])
q0 = np.array([np.pi/2, -np.pi/2, 0, -np.pi/2, 0, 0])  # Init joint position for the IK algorithm

q_calculated, all_delta_q, all_jacobians = inverse_kinematics_ur3e(X_B_TCP_target, q0)

<div class="alert alert-block alert-success"> Your IK solution should be shown in the MeshCat server, open the web page (<a href="http://localhost:7000">http://localhost:7000</a> or a different port) to see the simulation.</div>

In [ ]:
# Visualising convergence of the joint angles
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2)
fig.set_figheight(5)
fig.set_figwidth(15)

ax1.plot((np.rad2deg(np.array(all_delta_q))))
ax2.plot((np.rad2deg(np.array(all_delta_q))))
ax1.legend(plant.GetPositionNames(ur3e_idx))
ax2.legend(plant.GetPositionNames(ur3e_idx))
ax2.set_ylim([-0.5, 0.5])

<div class="alert alert-block alert-info">  <b>Task 6</b>: Use your FK function from Part 1 to verify if you obtained a correct solution.</div>

<div class="alert alert-block alert-info">  <b>Task 7</b>: Adjust $\alpha$ and try to find for which values the algorithm no longer converges. In addition, adjust the initial position q0 and see how this impacts the convergence path.</div>

Document your process here:

<div class="alert alert-block alert-info">  <b>Task 8</b>: Will the algorithm converge if we reduce the tolerance for the error $e$, i.e., if we set pos_tolerance_in_m = 0 and/or rot_tolerance_in_degrees = 0? Why or why not?</div>

Your answer:

<div class="alert alert-block alert-info">  <b>Task 9</b>: Reset your IK function parameters to proper values for $\alpha$, q0 and the tolerances. Move the real robot from the initial joint position q0 to the target and compare the path of the real robot with the positions that your virtual robot passed through. Are both paths the same?</div>

Your answer:

In [ ]:
# Re-run the optimisation and check the robot path in simulation
X_B_TCP_target = np.identity(4) 
X_B_TCP_target[0:3, 3] = np.array([-0.25, -0.05, 0.45])
q0 = np.array([np.pi/2, -np.pi/2, 0, -np.pi/2, 0, 0])  # Init joint position for the IK algorithm

q_calculated, all_delta_q, all_jacobians = inverse_kinematics_ur3e(X_B_TCP_target, q0)

In [ ]:
# See the robot path in real life
robot.move_to_joint_configuration(q0).wait()
robot.move_to_joint_configuration(q_calculated).wait(0.2)

### Inverse Jacobian Cartesian Planner
Let's define a path as an ordered set of configurations (in joint or task space).

<div class="alert alert-block alert-info">  <b>Task 10</b>: Does the inverse_kinematics_ur3e() function you implemented generate a path (i.e. an ordered set of joint configurations you can let the arm traverse to)? </div>


Your answer:

We will now use our inverse jacobian IK function to follow a path in Cartesian space. In Cartesian planning, the desired motion is typically defined as a trajectory in Cartesian space (e.g., moving the end-effector from point A to point B with a specific orientation). We will define three points and linearly interpolate between them. We do this for you in the next code cell and plot the desired end-effector position and velocities over time. 

In [ ]:
from pydrake.trajectories import PiecewisePose

# make a trajectory with three waypoints and linearly interpolate between them 
pose_traj = []

# first waypoint
X_B_TCP_target = np.identity(4)

X_B_TCP_target[0:3, 3] = np.array([-0.25, -0.05, 0.45])
pose_traj.append(X_B_TCP_target)

# second waypoint
X_B_TCP_target = X_B_TCP_target.copy()
X_B_TCP_target[0:3, 3] = np.array([-0.25, -0.42, 0.45])
pose_traj.append(X_B_TCP_target)

# third waypoint
X_B_TCP_target = X_B_TCP_target.copy()
X_B_TCP_target[0:3, 3] = np.array([-0.12, -0.25, 0.25])
pose_traj.append(X_B_TCP_target)

# add temporal dimensions (timings) to waypoints 
timing = [0, 5, 10]
dt = 0.1
t = np.arange(0, timing[-1], dt)
poses = []
for pose in pose_traj:
    poses.append(RigidTransform(pose))

# make the trajectory using your waypoints
traj_X_B_TCP = PiecewisePose.MakeLinear(timing, poses)

traj_p_B_G = traj_X_B_TCP.get_position_trajectory()

# Extract position values
p_B_TCP = traj_p_B_G.vector_values(t)

# Plot positions
plt.figure()  # Create a new figure for positions
plt.plot(t, p_B_TCP.T)
plt.title("End-Effector Position (p_B_TCP)")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.legend(["x", "y", "z"])
plt.grid(True)

# First derivative (velocity)
traj_V_G = traj_p_B_G.MakeDerivative()
p_VG = traj_V_G.vector_values(t)

# Plot velocities in a different plot
plt.figure()  # Create a new figure for velocities
plt.plot(t, p_VG.T, "--")
plt.title("End-Effector Velocity (V_B_TCP)")
plt.xlabel("Time [s]")
plt.ylabel("Velocity [m/s]")
plt.legend(["dx", "dy", "dz"])
plt.grid(True)

plt.show()


We will now sample this trajectory and calculate the IK result for each point of this sampled path.

Run the following code cell. It can take a couple of minutes, follow the progress in your meshcat instance. 

In [ ]:
# Initialize robot to starting configuration
robot_start_q = q_calculated.copy()

plant.SetPositions(plant.GetMyContextFromRoot(robot_context), ur3e_idx, robot_start_q)

# Force a diagram update to reflect the new configuration
diagram.ForcedPublish(robot_context)

# Time step for simulation
dt = 0.1  # 100ms time step
iterations = 0  # Track total iterations
ik_steps_per_iteration = []  # Store the number of IK substeps for each iteration
iterations_list = []
ik_steps_data = []  # Store substep count, q0, and q_calculated

# Main loop: iterate over time from 0 to 10 seconds, with a step size of dt
for t in np.arange(0, 10, dt):
    # Step 1: Get the desired pose (X_B_TCP) at time t from the trajectory
    X_B_TCP = traj_X_B_TCP.value(t)
    
    # Step 2: Get the current joint configuration (q0) of the robot
    q0 = plant.GetPositions(plant.GetMyContextFromRoot(robot_context), ur3e_idx)
    
    # Step 3: Perform inverse kinematics to compute the new joint configuration (q_calculated)
    q_calculated, all_delta_q, all_jacobians = inverse_kinematics_ur3e(
        X_B_TCP, q0, alpha=0.1, pos_tolerance_in_m=0.001, rot_tolerance_in_degrees=1
    )
    
    # Step 4: Log the number of substeps (iterations) the IK solver took
    ik_steps_per_iteration.append(len(all_delta_q))
    ik_steps_data.append({
        "substeps": len(all_delta_q),
        "q0": q0.copy(),
        "q_calculated": q_calculated.copy(),
        "jacobians": all_jacobians
    })
    
    # Step 5: Move the robot to the new joint configuration 
    robot.move_to_joint_configuration(q_calculated).wait(0.2)

    # Step 6: Increment the total iteration count
    iterations_list.append(iterations)
    iterations += 1

# Plot the number of IK steps required for each iteration over time
fig, ax1 = plt.subplots()

# First x-axis (time)
time_steps = np.arange(0, 10, dt)
ax1.plot(time_steps, ik_steps_per_iteration, label="IK substeps", color='b')
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Number of IK substeps")
ax1.set_title("Inverse Kinematics Substeps Required Over Time")
ax1.grid(True)
ax1.legend(loc="upper left")

# Second x-axis (iterations)
ax2 = ax1.twiny()  # Create a secondary x-axis
ax2.set_xlim(ax1.get_xlim())  # Match the limits of the original x-axis
ax2.set_xlabel("Iterations")  # Label for the second axis

# Select fewer iteration labels to avoid crowding the secondary x-axis
step_size = max(1, len(iterations_list) // 10)  # Show around 10 labels
reduced_iterations_list = iterations_list[::step_size]  # Reduce the iteration list
reduced_time_steps = time_steps[::step_size]  # Match the reduced iteration labels with time steps

ax2.set_xticks(reduced_time_steps)  # Use the reduced tick marks for the secondary axis
ax2.set_xticklabels(reduced_iterations_list)  # Set the reduced iteration labels

plt.show()

In the theory session, you saw that if the rank of the matrix drops belows 6 (i.e. it is not full row-rank anymore), the Jacobian is not invertible. We could check the rank of the Jacobian during our differential IK algorithm, however, we are more intrested when the matrix is starting to lose rank. A way to do this is by wachting the smallest singular value; as it approaches zero, the pseudo-inverse will blow up. We will plot this in the next cell for the previous exercise. 

In [ ]:
def plot_min_singular_values(all_jacobians):
    min_singular_values = []
    # Iterate over all Jacobians and compute the minimal singular value
    for J_G in all_jacobians:
        # Perform singular value decomposition without computing U and V matrices
        singular_values = np.linalg.svd(J_G, compute_uv=False)
        
        # Append the minimum singular value to the list
        min_singular_values.append(np.min(singular_values))
    
    # Plot the minimal singular values over the IK process
    plt.plot(min_singular_values)
    plt.xlabel("IK Iterations")
    plt.ylabel("Minimal Singular Value")
    plt.title("Minimal Singular Value of Jacobians During IK Process")
    plt.grid(True)
    plt.show()

all_jacobians = [ik_data["jacobians"] for ik_data in ik_steps_data if len(ik_data["jacobians"]) > 0]
plot_min_singular_values(all_jacobians)

Let's also look at the most expensive IK start and target positions. For every IK calculation that required relatively more iterations, we show the q_start and the q_calculated with one second in between. 

<div class="alert alert-block alert-info">  <b>Task 11</b>: Run the next cell and observe the meshcat visualizations (http://localhost:7000 or other port). Why do these IK calculations require more iterations, even though there is a small end-effector and joint displacement?</div>

In [ ]:
import time 

# Sort the IK results by the number of substeps (iterations) in descending order
sorted_ik_steps = sorted(ik_steps_data, key=lambda x: x["substeps"], reverse=True)

# Get the top 10 configurations that required the most IK iterations
top_10_ik_steps = sorted_ik_steps[:10]

for idx, ik_data in enumerate(top_10_ik_steps):
    print(f"Top {idx+1} IK configuration:")
    print(f"Initial configuration (q0): {ik_data['q0']}")
    print(f"Final configuration (q_calculated): {ik_data['q_calculated']}")
    print(f"Substeps taken: {ik_data['substeps']}")
    
    # Set the robot to the calculated joint configuration
    print("Showing initial configuration q0 on meshcat")
    plant.SetPositions(plant.GetMyContextFromRoot(robot_context), ur3e_idx, ik_data['q0'])
    diagram.ForcedPublish(robot_context)
    time.sleep(1)  
    print("Showing (calculated) goal configuration on meshcat")
    plant.SetPositions(plant.GetMyContextFromRoot(robot_context), ur3e_idx, ik_data['q_calculated'])
    diagram.ForcedPublish(robot_context)
    time.sleep(3)
    print("---- SHOWING NEXT EXPENSIVE CONFIGURATION IN MESHCAT -----")

Your answer: 

### Inverse kinematics: singularities

In the cell below, the simulated robot will attempt to move to a set of poses, each 3mm further towards negative x from the previous. You will see that the IK does not converge for any of the poses, even though the positions to be reached lie well within the workspace of the robot. However, due to the orientation restrictions, the pose is unreachable. These are examples of singularities in the robot's workspace.

In [ ]:
from pydrake.all import RotationMatrix, RollPitchYaw
import matplotlib.pyplot as plt

q0 = np.array([np.pi/2, -np.pi/2, 0, -np.pi/2, 0, 0])  # Init joint position for the IK algorithm

X_B_TCP_target = np.identity(4)
rpy = [-1.617, -0.556, -1.450] 

X_B_TCP_target[0:3,0:3] = RotationMatrix(RollPitchYaw(rpy)).matrix()
X_B_TCP_target[0, 3] = 0.241
X_B_TCP_target[1, 3] = 0.030 # tussen 0.030 en 0.012
X_B_TCP_target[2, 3] = 0.603

path_jacobians = None
for i in np.arange(0, 6):
    X_B_TCP_target[0, 3] = X_B_TCP_target[0, 3] - 0.002  # move in steps of 2 mm
    print(f"{i}: {X_B_TCP_target[:3,3]}")
    q_calculated, all_delta_q, all_jacobians = inverse_kinematics_ur3e(X_B_TCP_target, q0)

    # some jacobian bookkeeping 
    all_jacobians = np.array(all_jacobians)
    if path_jacobians is None:
        path_jacobians = all_jacobians
    else:
        path_jacobians = np.concatenate((path_jacobians, all_jacobians), axis=0)
    
fig, (ax1, ax2) = plt.subplots(1, 2)
fig.set_figheight(5)
fig.set_figwidth(15)

ax2.set_ylim([-0.5, 0.5])
ax1.plot((np.rad2deg(np.array(all_delta_q))))
ax2.plot((np.rad2deg(np.array(all_delta_q))))
ax1.legend(plant.GetPositionNames(ur3e_idx))
ax2.legend(plant.GetPositionNames(ur3e_idx))

plt.show()

plot_min_singular_values(path_jacobians)

<div class="alert alert-block alert-info">  <b>Task 12</b>: We will now attempt to reach the same poses on the real robot. How does the real robot behave differently from the simulated one? What could be the cause? </div>

Your answer:

In [ ]:
from pydrake.all import RotationMatrix, RollPitchYaw, AngleAxis

X_B_TCP_target = np.identity(4)
rpy = [-1.617, -0.556, -1.450] 

X_B_TCP_target[0:3,0:3] = RotationMatrix(RollPitchYaw(rpy)).matrix()
X_B_TCP_target[0, 3] = 0.241
X_B_TCP_target[1, 3] = 0.030 # tussen 0.030 en 0.012
X_B_TCP_target[2, 3] = 0.603


q0 = np.array([np.pi/2, -np.pi/2, 0, -np.pi/2, 0, 0])
robot.move_to_joint_configuration(q0).wait()
robot.move_to_tcp_pose(X_B_TCP_target).wait()

for i in np.arange(0, 6):
    X_B_TCP_target[0, 3] = X_B_TCP_target[0, 3] - 0.002  # move in steps of 1 mm
    print(f"{i}: {X_B_TCP_target[:,3]}")
    robot.move_to_tcp_pose(X_B_TCP_target).wait()

### Bonus: the pseudo-inverse Jacobian method for inverse kinematics

Go back to your inverse_kinematics_ur3e function and use the pseudo-inverse Jacobian instead of the transposed Jacobian.

<div class="alert alert-block alert-info">  <b>Task 13</b>: Compare the convergence trajectories of the pseudo-inverse method and the transpose method, which is better?</div>

Your answer:

# Wrapping up

Congratulations! You have now completed the second practical of the course.

A few questions to think about (note: these questions are not part of the practical submission):

* Given an IK solver that, for a specific TCP pose, outputs multiple viable joint positions. How would you select the joint position to assume?
* How could we adapt the robot's structure to reduce singularities?


Before shutting down the robot, please use freedrive to put it in a safe position (away from objects and do not let any links of robots rest outside the table so people do not collide against the robot when passing).

In [ ]:
robot.rtde_control.teachMode()

In [ ]:
robot.rtde_control.endTeachMode()